In [ ]:
#DailyChallenge

# 1. Charger et pretraiter l'ensemble de donnees MNIST
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Chargement du jeu de donnees
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalisation des pixels entre 0 et 1
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Encodage one-hot des etiquettes
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# Division deja faite (train/test)

# Affichage d'exemples d'images
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f'Label: {y_train[i]}')
    plt.axis('off')
plt.suptitle('Exemples d\'images MNIST')
plt.show()

# 2. Construire un reseau neuronal entierement connecte
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 3. Entrainer le reseau neuronal
history = model.fit(
    x_train, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# Visualisation des tendances de perte et precision
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Evolution de la perte')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Evolution de la precision')
plt.show()

# 4. Evaluer les performances du modele
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Precision sur l'ensemble de test: {test_accuracy:.4f}")

# Predictions
y_pred_proba = model.predict(x_test)
y_pred = np.argmax(y_pred_proba, axis=1)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Prediction')
plt.ylabel('Verite terrain')
plt.title('Matrice de confusion')
plt.show()

# Identification des chiffres avec lesquels le modele a le plus de difficultes
class_accuracy = []
for i in range(10):
    mask = (y_test == i)
    class_acc = np.mean(y_pred[mask] == y_test[mask])
    class_accuracy.append(class_acc)
    print(f"Chiffre {i}: precision = {class_acc:.4f}")

difficult_digits = np.argsort(class_accuracy)[:3]
print(f"\nChiffres avec lesquels le modele a le plus de difficultes: {difficult_digits}")

# Affichage d'exemples mal classes
misclassified_idx = np.where(y_pred != y_test)[0]
n_misclass = min(10, len(misclassified_idx))
plt.figure(figsize=(12, 4))
for i in range(n_misclass):
    idx = misclassified_idx[i]
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx], cmap='gray')
    plt.title(f'True: {y_test[idx]}\nPred: {y_pred[idx]}', color='red')
    plt.axis('off')
plt.suptitle('Exemples mal classes')
plt.show()